In [4]:
# we'll try to train the model with the NWD loss function for box regression and class prediction.
# The NWD loss is expected to improve the performance of the model, especially in scenarios where
# the bounding boxes vary significantly in size and aspect ratio.
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from models.deformable_detr import build_deformable_detr
from models.matcher import HungarianMatcher
from util.box_ops import NWD
from util.misc import NestedTensor, collate_fn
from datasets.coco import build_coco_dataset
import numpy as np
import time
import os


/Users/apple/Personal/Deformable-DETR/models/ops/modules/ms_deform_attn.py:79: SyntaxWarning: invalid escape sequence '\s'
  """
/Users/apple/Personal/Deformable-DETR/models/ops/modules/ms_deform_attn.py:79: SyntaxWarning: invalid escape sequence '\s'
  """


ModuleNotFoundError: No module named 'MultiScaleDeformableAttention'

In [ ]:
#Let's define the argument parser to include the NWD loss parameters
def get_args_parser():
    parser = argparse.ArgumentParser('Deformable DETR Detector', add_help=False)
    parser.add_argument('--lr', default=2e-4, type=float)
    parser.add_argument('--lr_backbone_names', default=["backbone.0"], type=str, nargs='+')
    parser.add_argument('--lr_backbone', default=2e-5, type=float)
    parser.add_argument('--lr_linear_proj_names', default=['reference_points', 'sampling_offsets'], type=str, nargs='+')
    parser.add_argument('--lr_linear_proj_mult', default=0.1, type=float)
    parser.add_argument('--batch_size', default=2, type=int)
    parser.add_argument('--weight_decay', default=1e-4, type=float)
    parser.add_argument('--epochs', default=50, type=int)
    parser.add_argument('--lr_drop', default=40, type=int)
    parser.add_argument('--lr_drop_epochs', default=None, type=int, nargs='+')
    parser.add_argument('--clip_max_norm', default=0.1, type=float,
                        help='gradient clipping max norm')


    parser.add_argument('--sgd', action='store_true')

    # # Variants of Deformable DETR
    # parser.add_argument('--with_box_refine', default=False, action='store_true')
    # parser.add_argument('--two_stage', default=False, action='store_true')

    # * Backbone
    parser.add_argument('--backbone', default='resnet50', type=str,
                        help="Name of the convolutional backbone to use")
    parser.add_argument('--dilation', action='store_true',
                        help="If true, we replace stride with dilation in the last convolutional block (DC5)")
    parser.add_argument('--position_embedding', default='sine', type=str, choices=('sine', 'learned'),
                        help="Type of positional embedding to use on top of the image features")
    parser.add_argument('--position_embedding_scale', default=2 * np.pi, type=float,
                        help="position / size * scale")
    parser.add_argument('--num_feature_levels', default=4, type=int, help='number of feature levels')

    # * Transformer
    #TODO: Change these to higher numbers for better performance
    parser.add_argument('--enc_layers', default=1, type=int,
                        help="Number of encoding layers in the transformer")
    parser.add_argument('--dec_layers', default=1, type=int,
                        help="Number of decoding layers in the transformer")
    parser.add_argument('--dim_feedforward', default=1024, type=int,
                        help="Intermediate size of the feedforward layers in the transformer blocks")
    parser.add_argument('--hidden_dim', default=256, type=int,
                        help="Size of the embeddings (dimension of the transformer)")
    parser.add_argument('--dropout', default=0.1, type=float,
                        help="Dropout applied in the transformer")
    parser.add_argument('--nheads', default=2, type=int,
                        help="Number of attention heads inside the transformer's attentions")
    parser.add_argument('--num_queries', default=300, type=int,
                        help="Number of query slots")
    parser.add_argument('--dec_n_points', default=4, type=int)
    parser.add_argument('--enc_n_points', default=4, type=int)

    # Loss
    parser.add_argument('--no_aux_loss', dest='aux_loss', action='store_false',
                        help="Disables auxiliary decoding losses (loss at each layer)")

    # * Matcher
    parser.add_argument('--set_cost_class', default=2, type=float,
                        help="Class coefficient in the matching cost")
    parser.add_argument('--set_cost_nwd', default=7, type=float,
                        help="nwd box coefficient in the matching cost")

    # * Loss coefficients
    parser.add_argument('--cls_loss_coef', default=2, type=float)
    parser.add_argument('--focal_alpha', default=0.25, type=float)
    parser.add_argument('--bbox_loss_nwd', default=7, type=float)


    # dataset parameters
    parser.add_argument('--dataset_file', default='coco')
    parser.add_argument('--coco_path', default='./data/coco', type=str)
    parser.add_argument('--remove_difficult', action='store_true')

    parser.add_argument('--output_dir', default='',
                        help='path where to save, empty for no saving')
    parser.add_argument('--device', default='cuda',
                        help='device to use for training / testing')
    parser.add_argument('--seed', default=42, type=int)
    parser.add_argument('--resume', default='', help='resume from checkpoint')
    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    parser.add_argument('--eval', action='store_true')
    parser.add_argument('--num_workers', default=2, type=int)
    parser.add_argument('--cache_mode', default=False, action='store_true', help='whether to cache images on memory')

    return parser

In [ ]:
#let's start by defining the detr model and run a simple training loop for one epoch as a sanity check.
def main(args):
    # Build the model
    model, criterion, postprocessors = build_deformable_detr(args)
    model.to(args.device)

    # Build the matcher
    matcher = HungarianMatcher(args)

    # Build the optimizer
    param_dicts = [
        {"params": [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]},
        {
            "params": [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad],
            "lr": args.lr_backbone,
        },
    ]
    optimizer = optim.AdamW(param_dicts, lr=args.lr, weight_decay=args.weight_decay)

    # Build the dataset and dataloader
    dataset_train = build_coco_dataset(image_set='train', args=args)
    # dataset_val = build_coco_dataset(image_set='val', args=args)

    data_loader_train = DataLoader(dataset_train, batch_size=args.batch_size, shuffle=True,
                                   num_workers=args.num_workers, collate_fn=collate_fn)
    # data_loader_val = DataLoader(dataset_val, batch_size=args.batch_size, shuffle=False,
    #                              num_workers=args.num_workers, collate_fn=collate_fn)

    # Training loop for one epoch
    model.train()
    for samples, targets in data_loader_train:
        samples = samples.to(args.device)
        targets = [{k: v.to(args.device) for k, v in t.items()} for t in targets]

        outputs = model(samples)

        loss_dict = criterion(outputs, targets)
        losses = sum(loss_dict[k] * criterion.weight_dict[k] for k in loss_dict.keys() if k in criterion.weight_dict)

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        #Print the loss for monitoring
        print(f"Loss: {losses.item()}")

    print("Training completed for one epoch.")

In [ ]:
#Define the arguments to per parsed and start the main function
if __name__ == '__main__':
    parser = argparse.ArgumentParser('Deformable DETR training and evaluation script', parents=[get_args_parser()])
    args = parser.parse_args()
    main(args)